[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/076.%20The%20VRP%20with%20Split%20Deliveries%20%28SDVRP%29/P76-Tier-8_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 76. The VRP with Split Deliveries (SDVRP)
## Tier 8: The Value-Aligned and Ethical Framework

### Purpose and scope

This notebook evaluates a small set of feasible split-delivery route plans using stakeholder utility, ethical indicators, and a simple compliance and risk review. It keeps the SDVRP setting: a customer may receive portions of their demand from more than one vehicle. The example's total demand exceeds its total vehicle capacity, so complete service is impossible with the stated fleet. Candidate plans therefore respect vehicle capacities and serve only part of total demand; service levels make that limitation visible.

The procedure is a transparent, finite-candidate evaluation—not a complete SDVRP search and not a proof of global optimality. Its purpose is to show how a baseline and a few specified alternatives can be assessed within a value-aligned framework.

### Inputs and outputs

- **Inputs:** depot and customer coordinates, customer demands, vehicle capacities, stakeholder weights, ethical-score weights, and stated operational assumptions.
- **Outputs:** the highest-scoring plan among the evaluated candidates, its distance and service metrics, stakeholder utilities, ethical indicators, compliance findings, and a heuristic risk score.
- **Scope:** Euclidean distances represent travel distance in model units. The example does not contain customer time windows, actual costs, emissions measurements, noise observations, maintenance records, or a time-of-day schedule. Indicators that depend on missing operational data are treated as explicit assumptions or are not assessed; they are not presented as measured facts or legal determinations.

### Method and mathematical representation

Each candidate consists of vehicle routes and delivery quantities. Let $K$ be the set of vehicles, $C$ the set of customers, $q_i$ the demand of customer $i$, and $Q_k$ the capacity of vehicle $k$. Let $x_{ki}$ denote the quantity delivered to customer $i$ by vehicle $k$. A candidate must satisfy

$$
0 \leq \sum_{k \in K} x_{ki} \leq q_i \quad (i \in C),
\qquad
\sum_{i \in C} x_{ki} \leq Q_k \quad (k \in K).
$$

The first condition prevents over-delivery; the second enforces vehicle capacity. Split delivery is allowed because more than one $x_{ki}$ may be positive for the same customer. A route's distance is the sum of Euclidean distances between consecutive nodes, including the outbound and return depot legs. In the code, these checks and distance calculations are performed by `ValueAlignedOptimizer` using the `deliveries`, `routes`, `capacities`, and `distance_matrix` data.

For customer $i$, the service fraction is $s_i = (\sum_k x_{ki})/q_i$ when $q_i > 0$. The fairness indicator is $F = 1 - \operatorname{Var}(s_i)$ across customers. This measures equality of service fractions, not whether service is sufficient: equal but low service can still receive a high fairness score. The sustainability indicator uses the stated emissions factor $e=0.5$ kg CO$_2$ per distance unit and is $S = \max(0, 1 - eD/1000)$, where $D$ is total route distance.

The ethical score is a weighted sum of fairness, sustainability, transparency, and accountability indicators:

$$
E = 0.25F + 0.20S + 0.15T + 0.15A.
$$

Here $T$ and $A$ are explicitly assumed documentation and traceability scores. The weights sum to $0.75$, so $E$ is not itself a 0-to-1 normalized score. The candidate-ranking rule normalizes it by $0.75$ before applying the ethical multiplier. For each candidate $r$, the ranking score is

$$
R_r = U_r\left(0.7 + 0.3\frac{E_r}{0.75}\right),
$$

where $U_r$ is the sum of the five stakeholder utilities. The framework evaluates the baseline and three explicitly specified alternatives and selects the candidate with the largest $R_r$. This finite comparison takes time proportional to the number of candidates times the work needed to evaluate their routes and deliveries; it does not enumerate all possible SDVRP plans.

The flow follows the example's value-aligned approach: identify stakeholders, define utility measures, evaluate ethical indicators, compare candidate plans, and report compliance and risk. The scores are decision-support indicators based on the assumptions above; they do not replace legal review, measured environmental data, or a comprehensive optimization model.

In [ ]:
# NumPy supports distance and metric calculations; Matplotlib visualizes results.
import numpy as np
import matplotlib.pyplot as plt
from math import isclose, sqrt

# Keep the example reproducible if randomized experiments are added later.
np.random.seed(76)

print("Libraries imported successfully for the value-aligned SDVRP example.")

In [ ]:
# Store the instance data together so that route, distance, and capacity checks
# all refer to the same explicitly supplied problem definition.
class SDVRPInstance:
    """Represent an SDVRP instance with Euclidean travel distances."""

    def __init__(self, depot_coords, customer_coords, demands, vehicle_capacities):
        if len(customer_coords) != len(demands):
            raise ValueError("Each customer must have exactly one demand value.")
        if any(demand < 0 for demand in demands):
            raise ValueError("Customer demands must be non-negative.")
        if any(capacity < 0 for capacity in vehicle_capacities):
            raise ValueError("Vehicle capacities must be non-negative.")

        self.depot = depot_coords
        self.customers = customer_coords
        self.demands = demands
        self.capacities = vehicle_capacities
        self.n_customers = len(customer_coords)
        self.n_vehicles = len(vehicle_capacities)

    def calculate_distance_matrix(self):
        """Return the symmetric Euclidean distance matrix for all nodes."""
        all_nodes = [self.depot] + self.customers
        n_nodes = len(all_nodes)
        distances = np.zeros((n_nodes, n_nodes), dtype=float)

        for i in range(n_nodes):
            for j in range(n_nodes):
                x1, y1 = all_nodes[i]
                x2, y2 = all_nodes[j]
                distances[i, j] = sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

        return distances


# Customer node IDs are 1 through 4; node 0 is the depot.
depot_coords = (0, 0)
customer_coords = [(10, 15), (20, 5), (15, 20), (25, 10)]
demands = [70, 130, 60, 80]
vehicle_capacities = [100, 100]

instance = SDVRPInstance(
    depot_coords,
    customer_coords,
    demands,
    vehicle_capacities,
)
distance_matrix = instance.calculate_distance_matrix()

print("SDVRP instance:")
print(f"- Depot: {depot_coords}")
print(f"- Customers: {instance.n_customers} with demands: {demands}")
print(f"- Vehicles: {instance.n_vehicles} with capacities: {vehicle_capacities}")
print(f"- Total demand: {sum(demands)} units")
print(f"- Total capacity: {sum(vehicle_capacities)} units")
print("- Full demand service is infeasible with the stated total capacity.")

## Stakeholders and utility measures

A stakeholder utility is a weighted summary of indicators relevant to one group. Utility values here are comparative scores built from the available instance data and stated assumptions; they are not monetary valuations or empirical measurements. Customer service and access, employee workload balance, environmental distance-related proxies, company service and efficiency, and regulatory indicators provide the five perspectives used in the candidate ranking.

The implementation accepts the current candidate's metrics as an argument. This keeps each evaluation tied to the candidate being scored rather than silently reusing metrics from another plan.

In [ ]:
class StakeholderAnalysis:
    """Calculate transparent, assumption-based stakeholder utilities."""

    def __init__(self, instance):
        self.instance = instance
        self.stakeholders = {
            "customers": {
                "priority": 0.3,
                "cost_sensitivity": 0.2,
                "accessibility": 0.25,
                "loyalty": 0.15,
            },
            "employees": {
                "workload_balance": 0.4,
                "safety": 0.5,
                "job_satisfaction": 0.3,
                "training_needs": 0.2,
            },
            "environment": {
                "emissions": 0.3,
                "noise_pollution": 0.2,
                "energy_efficiency": 0.4,
                "sustainability": 0.3,
            },
            "company": {
                "profitability": 0.6,
                "market_share": 0.3,
                "reputation": 0.2,
                "growth": 0.1,
            },
            "regulators": {
                "compliance": 0.7,
                "safety_standards": 0.8,
                "public_interest": 0.6,
                "transparency": 0.5,
            },
        }

    def calculate_stakeholder_utility(self, solution, stakeholder, metrics):
        """Return one stakeholder's weighted score for a candidate plan."""
        weights = self.stakeholders[stakeholder]

        if stakeholder == "customers":
            cost_utility = 1 / (
                1 + metrics["distance_per_unit"] * weights["cost_sensitivity"]
            )
            access_utility = 1 / (
                1
                + metrics["avg_distance_per_delivery"]
                / 50
                * weights["accessibility"]
            )
            return (
                weights["priority"] * metrics["service_level"]
                + weights["cost_sensitivity"] * cost_utility
                + weights["accessibility"] * access_utility
                + weights["loyalty"] * metrics["service_level"]
            )

        if stakeholder == "employees":
            workload_utility = 1 / (
                1 + metrics["workload_variance"] * weights["workload_balance"]
            )
            return (
                weights["workload_balance"] * workload_utility
                + weights["safety"] * metrics["assumed_safety_score"]
                + weights["job_satisfaction"] * workload_utility
            )

        if stakeholder == "environment":
            emissions_utility = 1 / (
                1 + metrics["total_emissions"] / 100 * weights["emissions"]
            )
            energy_utility = 1 / (
                1 + metrics["fuel_proxy"] / 1000 * weights["energy_efficiency"]
            )
            return (
                weights["emissions"] * emissions_utility
                + weights["noise_pollution"] * metrics["assumed_noise_score"]
                + weights["energy_efficiency"] * energy_utility
                + weights["sustainability"] * metrics["sustainability_score"]
            )

        if stakeholder == "company":
            return (
                weights["profitability"] * metrics["transport_efficiency"]
                + weights["market_share"] * metrics["service_level"]
                + weights["reputation"] * metrics["transparency_score"]
                + weights["growth"] * metrics["service_level"]
            )

        if stakeholder == "regulators":
            return (
                weights["compliance"] * metrics["regulatory_compliance"]
                + weights["safety_standards"] * metrics["assumed_safety_score"]
                + weights["public_interest"] * metrics["service_level"]
                + weights["transparency"] * metrics["transparency_score"]
            )

        raise ValueError(f"Unknown stakeholder: {stakeholder}")

    def _calculate_service_level(self, solution, customer_id):
        """Return the fraction of one customer's demand that is delivered."""
        customer_index = customer_id - 1
        demand = self.instance.demands[customer_index]
        delivered = sum(
            vehicle_deliveries.get(customer_id, 0)
            for vehicle_deliveries in solution["deliveries"]
        )
        if demand == 0:
            return 1.0
        return delivered / demand

    def _calculate_service_level(self, solution, metrics, customer_id=None):
        """Return total service or one customer's service fraction."""
        if customer_id is not None:
            return self._service_fraction(solution, customer_id)
        return metrics["service_level"]

    def _service_fraction(self, solution, customer_id):
        customer_index = customer_id - 1
        demand = self.instance.demands[customer_index]
        delivered = sum(
            vehicle_deliveries.get(customer_id, 0)
            for vehicle_deliveries in solution["deliveries"]
        )
        if demand == 0:
            return 1.0
        return delivered / demand


stakeholder_analysis = StakeholderAnalysis(instance)
print("Stakeholder utility measures are ready for candidate evaluation.")

## Ethical indicators and candidate ranking

The ethical indicators are measured separately so that their meaning remains inspectable. Fairness compares customer service fractions; sustainability uses the declared distance-based emissions proxy; transparency and accountability use stated illustrative assumptions. These indicators contribute to a score, but they are not hard feasibility constraints. Optional custom checks can be registered separately and evaluated against a threshold.

The ranking step scores the baseline and each specified alternative with the same stakeholder and ethical calculations. Because only a finite candidate set is considered, the selected plan is best within that set, not necessarily globally optimal.

In [ ]:
class EthicalFramework:
    """Evaluate ethical indicators and optional custom compliance checks."""

    def __init__(self, instance, stakeholder_analysis):
        self.instance = instance
        self.stakeholder_analysis = stakeholder_analysis
        self.constraints = []
        self.principles = {
            "fairness": {"weight": 0.25},
            "sustainability": {"weight": 0.20},
            "transparency": {"weight": 0.15},
            "accountability": {"weight": 0.15},
        }
        self.transparency_assumption = 0.8
        self.accountability_assumption = 0.9

    def _fairness_constraint(self, solution, metrics):
        """Score equality of customer service fractions, not service adequacy."""
        service_levels = [
            self.stakeholder_analysis._service_fraction(solution, customer_id)
            for customer_id in range(1, self.instance.n_customers + 1)
        ]
        if len(service_levels) <= 1:
            return 1.0
        return max(0.0, 1.0 - float(np.var(service_levels)))

    def _sustainability_constraint(self, solution, metrics):
        """Score the distance-based emissions proxy on a 0-to-1 scale."""
        return max(0.0, 1.0 - metrics["total_emissions"] / 1000)

    def _transparency_constraint(self, solution, metrics):
        """Use the stated documentation-quality assumption as an indicator."""
        return self.transparency_assumption

    def _accountability_constraint(self, solution, metrics):
        """Use the stated traceability assumption as an indicator."""
        return self.accountability_assumption

    def score_components(self, solution, metrics):
        """Return the four component scores used in the ethical score."""
        return {
            "fairness": self._fairness_constraint(solution, metrics),
            "sustainability": self._sustainability_constraint(solution, metrics),
            "transparency": self._transparency_constraint(solution, metrics),
            "accountability": self._accountability_constraint(solution, metrics),
        }

    def evaluate_solution(self, solution, metrics):
        """Return the weighted ethical score; its maximum is the weight sum."""
        scores = self.score_components(solution, metrics)
        return sum(
            self.principles[principle]["weight"] * scores[principle]
            for principle in self.principles
        )

    def add_constraint(self, principle, implementation_function):
        """Register an optional custom check returning a numeric score."""
        self.constraints.append(
            {
                "principle": principle,
                "implementation": implementation_function,
            }
        )

    def check_compliance(self, solution, metrics, threshold=0.7):
        """Check registered custom constraints against a score threshold."""
        for constraint in self.constraints:
            score = constraint["implementation"](solution, metrics)
            if score < threshold:
                return False, f"Constraint threshold not met: {constraint['principle']}"
        return True, "All registered custom constraints meet the threshold."


ethical_framework = EthicalFramework(instance, stakeholder_analysis)
print("Ethical indicators and optional custom checks are ready.")

In [ ]:
class ValueAlignedOptimizer:
    """Compare a baseline and specified alternatives by weighted utility."""

    def __init__(self, instance, stakeholder_analysis, ethical_framework):
        self.instance = instance
        self.stakeholder_analysis = stakeholder_analysis
        self.ethical_framework = ethical_framework
        self.distance_matrix = instance.calculate_distance_matrix()

    def validate_solution(self, solution):
        """Check route structure, delivery bounds, and vehicle capacities."""
        if len(solution["routes"]) != self.instance.n_vehicles:
            raise ValueError("Each vehicle must have one route in this example.")
        if len(solution["deliveries"]) != self.instance.n_vehicles:
            raise ValueError("Each vehicle must have one delivery record.")

        delivered_by_customer = [0.0] * self.instance.n_customers
        for vehicle_index, (route, deliveries) in enumerate(
            zip(solution["routes"], solution["deliveries"])
        ):
            if len(route) < 3 or route[0] != 0 or route[-1] != 0:
                raise ValueError("Each route must start and end at depot node 0.")
            if len(route[1:-1]) != len(set(route[1:-1])):
                raise ValueError("A route cannot visit the same customer twice.")
            if any(node < 1 or node > self.instance.n_customers for node in route[1:-1]):
                raise ValueError("A route contains an unknown customer node.")

            route_customers = set(route[1:-1])
            vehicle_load = 0.0
            for customer_id, quantity in deliveries.items():
                if customer_id not in route_customers:
                    raise ValueError("A vehicle cannot deliver to an unvisited customer.")
                if quantity <= 0:
                    raise ValueError("Recorded delivery quantities must be positive.")
                vehicle_load += quantity
                delivered_by_customer[customer_id - 1] += quantity

            if vehicle_load > self.instance.capacities[vehicle_index] + 1e-9:
                raise ValueError("A vehicle's delivery load exceeds its capacity.")

        for customer_index, delivered in enumerate(delivered_by_customer):
            if delivered > self.instance.demands[customer_index] + 1e-9:
                raise ValueError("A customer's total delivery exceeds demand.")

        return True

    def calculate_metrics(self, solution):
        """Calculate candidate metrics from its routes and delivery quantities."""
        self.validate_solution(solution)
        total_distance = sum(
            self.distance_matrix[origin, destination]
            for route in solution["routes"]
            for origin, destination in zip(route[:-1], route[1:])
        )
        delivered_by_customer = [0.0] * self.instance.n_customers
        vehicle_loads = []
        delivery_stops = 0

        for deliveries in solution["deliveries"]:
            vehicle_loads.append(sum(deliveries.values()))
            delivery_stops += len(deliveries)
            for customer_id, quantity in deliveries.items():
                delivered_by_customer[customer_id - 1] += quantity

        total_delivered = sum(delivered_by_customer)
        total_demand = sum(self.instance.demands)
        service_level = total_delivered / total_demand if total_demand else 1.0
        distance_per_unit = total_distance / total_delivered if total_delivered else 0.0
        avg_distance_per_delivery = (
            total_distance / delivery_stops if delivery_stops else 0.0
        )
        utilization = [
            load / capacity if capacity else 0.0
            for load, capacity in zip(vehicle_loads, self.instance.capacities)
        ]
        workload_variance = float(np.var(utilization)) if utilization else 0.0
        total_emissions = total_distance * 0.5
        sustainability_score = max(0.0, 1.0 - total_emissions / 1000)
        transport_efficiency = 1 / (1 + distance_per_unit)

        # These scores are assumptions because the instance has no measured
        # safety, noise, maintenance, or time-of-day observations.
        assumed_safety_score = 1.0
        assumed_noise_score = 1.0
        transparency_score = self.ethical_framework.transparency_assumption

        # Compliance uses only the example's assessable single-period checks.
        estimated_hours = total_distance / 50
        driving_compliant = estimated_hours <= 8
        emissions_rate_compliant = 0.5 <= 0.5
        maintenance_compliant = 15 <= 30
        regulatory_compliance = sum(
            [driving_compliant, emissions_rate_compliant, maintenance_compliant]
        ) / 3

        return {
            "total_distance": total_distance,
            "total_delivered": total_delivered,
            "total_demand": total_demand,
            "service_level": service_level,
            "distance_per_unit": distance_per_unit,
            "avg_distance_per_delivery": avg_distance_per_delivery,
            "vehicle_loads": vehicle_loads,
            "vehicle_utilization": (
                sum(vehicle_loads) / sum(self.instance.capacities)
                if sum(self.instance.capacities)
                else 0.0
            ),
            "workload_variance": workload_variance,
            "total_emissions": total_emissions,
            "fuel_proxy": total_distance * 0.3,
            "sustainability_score": sustainability_score,
            "transport_efficiency": transport_efficiency,
            "assumed_safety_score": assumed_safety_score,
            "assumed_noise_score": assumed_noise_score,
            "transparency_score": transparency_score,
            "regulatory_compliance": regulatory_compliance,
        }

    def optimize_with_ethics(self, base_solution):
        """Select the highest-scoring plan from the specified finite candidates."""
        print("=== VALUE-ALIGNED CANDIDATE EVALUATION ===")

        candidates = [base_solution] + self._generate_ethical_alternatives(base_solution)
        best_solution = None
        best_rank = float("-inf")
        best_details = None

        for candidate_index, solution in enumerate(candidates):
            metrics = self.calculate_metrics(solution)
            ethical_score = self.ethical_framework.evaluate_solution(solution, metrics)
            stakeholder_utilities = {
                stakeholder: self.stakeholder_analysis.calculate_stakeholder_utility(
                    solution,
                    stakeholder,
                    metrics,
                )
                for stakeholder in self.stakeholder_analysis.stakeholders
            }
            total_utility = sum(stakeholder_utilities.values())
            ethical_weight_sum = sum(
                principle["weight"]
                for principle in self.ethical_framework.principles.values()
            )
            normalized_ethical_score = (
                ethical_score / ethical_weight_sum if ethical_weight_sum else 0.0
            )
            rank = total_utility * (0.7 + 0.3 * normalized_ethical_score)

            # Keep the first candidate in an exact or near tie for deterministic reporting.
            if best_solution is None or rank > best_rank and not isclose(
                rank, best_rank, rel_tol=1e-9, abs_tol=1e-12
            ):
                best_solution = solution
                best_rank = rank
                best_details = {
                    "metrics": metrics,
                    "ethical_score": ethical_score,
                    "ethical_components": self.ethical_framework.score_components(
                        solution, metrics
                    ),
                    "stakeholder_utilities": stakeholder_utilities,
                    "total_utility": total_utility,
                    "ranking_score": rank,
                    "candidate_index": candidate_index,
                }

        print(f"Evaluated candidates: {len(candidates)}")
        print(f"Selected candidate index: {best_details['candidate_index']}")
        print(f"Selected distance: {best_details['metrics']['total_distance']:.2f} distance units")
        print(f"Selected service level: {best_details['metrics']['service_level']:.3f}")
        print(f"Ethical score: {best_details['ethical_score']:.3f} (maximum 0.750)")
        print(f"Stakeholder utility: {best_details['total_utility']:.3f}")
        print(f"Combined ranking score: {best_details['ranking_score']:.3f}")

        return best_solution, best_details

    def _generate_ethical_alternatives(self, base_solution):
        """Return three explicit, capacity-feasible split-delivery alternatives."""
        alternatives = [
            {
                "routes": [[0, 2, 0], [0, 4, 3, 0]],
                "deliveries": [{2: 100}, {4: 80, 3: 20}],
                "method": "priority_customer_candidate",
            },
            {
                "routes": [[0, 1, 3, 0], [0, 2, 0]],
                "deliveries": [{1: 70, 3: 30}, {2: 100}],
                "method": "distance_candidate",
            },
            {
                "routes": [[0, 1, 4, 0], [0, 2, 3, 0]],
                "deliveries": [{1: 70, 4: 30}, {2: 80, 3: 20}],
                "method": "workload_and_service_candidate",
            },
        ]
        return alternatives


optimizer = ValueAlignedOptimizer(
    instance,
    stakeholder_analysis,
    ethical_framework,
)
print("Candidate evaluation is ready; each plan will be checked against the instance.")

## Compliance and operational risk

Compliance checks must distinguish a violation from information that is unavailable. This example can estimate driving time from total route distance using the stated average speed, check the stated emissions rate, and apply the stated maintenance interval assumption. It cannot determine annual emissions from a single planning instance or assess nighttime noise without a schedule and observations, so those matters are reported as outside the measurable scope rather than assumed violations.

The risk score is a heuristic product of normalized risk factors. It is an indicator for comparing plans under the stated assumptions, not a probability of failure.

In [ ]:
class ComplianceMonitor:
    """Check assessable example requirements and calculate a heuristic risk score."""

    def __init__(self, instance):
        self.instance = instance
        self.distance_matrix = instance.calculate_distance_matrix()
        self.regulations = {
            "driving_hours": {"max_daily_hours": 8, "min_rest_time": 0.5},
            "vehicle_maintenance": {
                "inspection_interval_days": 30,
                "safety_checks": True,
            },
            "emissions": {"max_co2_per_km": 0.5, "annual_limit": 10000},
            "noise": {"max_db_level": 70, "night_restrictions": True},
        }
        self.violations = []
        self.risk_assessment = {}

    def check_compliance(self, solution, metrics):
        """Return the assessable-check score and any observed violations."""
        compliance_issues = []
        assessed_checks = 0

        estimated_hours = metrics["total_distance"] / 50
        assessed_checks += 1
        if estimated_hours > self.regulations["driving_hours"]["max_daily_hours"]:
            compliance_issues.append("Estimated driving time exceeds the daily limit.")

        emissions_per_distance_unit = 0.5
        assessed_checks += 1
        if emissions_per_distance_unit > self.regulations["emissions"]["max_co2_per_km"]:
            compliance_issues.append("The assumed emissions rate exceeds the stated limit.")

        # The 15-day maintenance interval is an explicit illustrative assumption.
        days_since_last_maintenance = 15
        assessed_checks += 1
        if (
            days_since_last_maintenance
            > self.regulations["vehicle_maintenance"]["inspection_interval_days"]
        ):
            compliance_issues.append("The assumed maintenance interval is overdue.")

        self.violations = list(compliance_issues)
        compliance_score = (
            (assessed_checks - len(compliance_issues)) / assessed_checks
            if assessed_checks
            else 1.0
        )
        return compliance_score, compliance_issues

    def assess_risks(self, solution, metrics):
        """Return a normalized heuristic score; this is not a failure probability."""
        route_complexity = min(
            1.0,
            len(solution["routes"]) / max(1, self.instance.n_vehicles),
        )
        risk_factors = {
            "route_complexity": (route_complexity, 0.20),
            "vehicle_utilization": (metrics["vehicle_utilization"], 0.20),
            "demand_volatility": (0.3, 0.20),
            "weather_sensitivity": (0.2, 0.20),
            "traffic_conditions": (0.1, 0.20),
        }

        risk_score = 1.0
        for factor_value, weight in risk_factors.values():
            risk_score *= 1 - factor_value * weight

        self.risk_assessment = {
            name: value for name, (value, _) in risk_factors.items()
        }
        return risk_score


compliance_monitor = ComplianceMonitor(instance)
print("Compliance and risk checks are ready for the selected candidate.")

In [ ]:
# The baseline follows the stated node numbering and splits customer 2's
# delivery between vehicles while respecting both capacity limits.
base_solution = {
    "routes": [[0, 1, 2, 0], [0, 2, 3, 4, 0]],
    "deliveries": [{1: 70, 2: 30}, {2: 30, 3: 60, 4: 10}],
    "method": "baseline",
}

base_metrics = optimizer.calculate_metrics(base_solution)
ethical_solution, selected_details = optimizer.optimize_with_ethics(base_solution)
ethical_metrics = optimizer.calculate_metrics(ethical_solution)
compliance_score, compliance_issues = compliance_monitor.check_compliance(
    ethical_solution,
    ethical_metrics,
)
risk_score = compliance_monitor.assess_risks(ethical_solution, ethical_metrics)

results = {
    "base_solution": base_solution,
    "base_metrics": base_metrics,
    "ethical_solution": ethical_solution,
    "ethical_metrics": ethical_metrics,
    "ethical_components": selected_details["ethical_components"],
    "stakeholder_utilities": selected_details["stakeholder_utilities"],
    "compliance_score": compliance_score,
    "compliance_issues": compliance_issues,
    "risk_score": risk_score,
}

print("\n=== SELECTED PLAN REVIEW ===")
print(f"Baseline distance: {base_metrics['total_distance']:.2f} distance units")
print(f"Selected distance: {ethical_metrics['total_distance']:.2f} distance units")
print(f"Baseline service level: {base_metrics['service_level']:.3f}")
print(f"Selected service level: {ethical_metrics['service_level']:.3f}")
print(f"Compliance score on assessable checks: {compliance_score:.2f}")
print(f"Heuristic risk score: {risk_score:.3f}")
if compliance_issues:
    print("Assessable compliance findings:")
    for issue in compliance_issues:
        print(f"- {issue}")
else:
    print("No violations were identified among the assessable checks.")
print("Annual emissions and noise restrictions are outside this instance's measurable scope.")

# The plan checks ensure reported totals agree with the candidate data.
assert optimizer.validate_solution(ethical_solution)
assert isclose(
    ethical_metrics["total_delivered"],
    sum(sum(deliveries.values()) for deliveries in ethical_solution["deliveries"]),
)
assert isclose(
    ethical_metrics["total_distance"],
    sum(
        distance_matrix[origin, destination]
        for route in ethical_solution["routes"]
        for origin, destination in zip(route[:-1], route[1:])
    ),
)

## Visualizing the evaluated results

The plots use values calculated from the selected plan and the baseline. Ethical-principle bars display the selected plan's component scores, while stakeholder bars display utilities calculated from its own metrics. The vertical scales and distance units are labeled so that the figures can be interpreted without treating illustrative scores as measured outcomes.

In [ ]:
def visualize_ethical_results(results):
    """Plot distance, assessable compliance, ethical indicators, and utilities."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    methods = ["Baseline", "Selected candidate"]
    distances = [
        results["base_metrics"]["total_distance"],
        results["ethical_metrics"]["total_distance"],
    ]
    bars = ax1.bar(methods, distances, color=["steelblue", "seagreen"], alpha=0.8)
    ax1.set_ylabel("Total route distance (model units)")
    ax1.set_title("Route Distance")
    ax1.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, distances):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    compliance_values = [
        results["compliance_score"],
        results["risk_score"],
    ]
    compliance_labels = ["Assessable compliance", "Heuristic risk"]
    bars = ax2.bar(
        compliance_labels,
        compliance_values,
        color=["royalblue", "darkorange"],
        alpha=0.8,
    )
    ax2.set_ylabel("Score (0 to 1)")
    ax2.set_title("Compliance and Risk Indicators")
    ax2.set_ylim(0, 1)
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, compliance_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    principles = list(results["ethical_components"])
    principle_scores = [
        results["ethical_components"][principle] for principle in principles
    ]
    bars = ax3.bar(principles, principle_scores, color="mediumpurple", alpha=0.8)
    ax3.set_ylabel("Indicator score (0 to 1)")
    ax3.set_title("Ethical Indicators for the Selected Candidate")
    ax3.set_ylim(0, 1)
    ax3.tick_params(axis="x", rotation=20)
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, principle_scores):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    stakeholder_names = list(results["stakeholder_utilities"])
    utilities = [
        results["stakeholder_utilities"][name] for name in stakeholder_names
    ]
    ax4.bar(stakeholder_names, utilities, color="teal", alpha=0.8)
    ax4.set_ylabel("Weighted utility (illustrative score)")
    ax4.set_title("Stakeholder Utilities for the Selected Candidate")
    ax4.tick_params(axis="x", rotation=20)
    ax4.grid(True, axis="y", alpha=0.3)

    fig.suptitle("Value-Aligned SDVRP Candidate Evaluation", fontsize=16)
    plt.tight_layout()
    plt.show()


visualize_ethical_results(results)

## Interpretation, strengths, and limitations

The selected plan is the highest-ranked member of the evaluated candidate set under the declared weights and assumptions. Its distance, service level, ethical components, stakeholder utilities, compliance checks, and heuristic risk score describe different aspects of that comparison; none alone establishes that a plan is universally fair, sustainable, safe, or legally compliant. In particular, fairness measures equality of service fractions, while the aggregate service level measures how much total demand is served.

**Strengths**

- Makes stakeholder weights and ethical indicators visible.
- Checks candidate delivery quantities against demand and vehicle capacity.
- Reports measured-from-instance quantities separately from illustrative assumptions.
- Provides a reproducible, inspectable comparison of a baseline and specified alternatives.

**Limitations**

- The finite candidate set is not an exhaustive search and gives no global optimality guarantee.
- The example's total capacity is insufficient for complete service, so its candidates necessarily leave some demand unmet.
- Utility weights, documentation and traceability scores, emissions factor, average speed, maintenance timing, and risk-factor assumptions require domain validation before operational use.
- Time windows, actual operating costs, measured emissions, noise, annual activity, and detailed regulatory rules are not modeled.

### When this framework is useful

This type of evaluation can help structure discussions in regulated or sustainability-focused operations where decision-makers need to make trade-offs explicit. An operational deployment would require validated data, stakeholder-approved weights, relevant legal expertise, and a candidate-generation process appropriate to the scale of the routing problem. Those additions are outside the implementation in this notebook.

In [ ]:
print("=== VALUE-ALIGNED SDVRP SUMMARY ===")
print()
print("1. Candidate performance:")
print(f"   Baseline distance: {results['base_metrics']['total_distance']:.2f} model units")
print(f"   Selected distance: {results['ethical_metrics']['total_distance']:.2f} model units")
print(f"   Baseline service level: {results['base_metrics']['service_level']:.3f}")
print(f"   Selected service level: {results['ethical_metrics']['service_level']:.3f}")
print()

print("2. Ethical indicators for the selected candidate:")
for principle, score in results["ethical_components"].items():
    print(f"   - {principle.title()}: {score:.3f}")
print()

print("3. Compliance and risk:")
print(f"   Assessable compliance score: {results['compliance_score']:.2f}")
print(f"   Heuristic risk score: {results['risk_score']:.3f}")
if results["compliance_issues"]:
    for issue in results["compliance_issues"]:
        print(f"   - Finding: {issue}")
else:
    print("   - No violations were identified among the assessable checks.")
print("   - Annual emissions and noise compliance are not determined by this example.")
print()

print("4. Interpretation:")
print("   - The ranking compares only the baseline and three specified alternatives.")
print("   - The stated fleet cannot serve all demand because total capacity is insufficient.")
print("   - Scores depend on the declared weights, proxies, and operational assumptions.")
print("   - The result is decision support, not a global optimality or legal-compliance claim.")